In [4]:
import pandas as pd 

df = pd.read_csv(r"D:\Python project\PROJECT\data\processed\cicids2017_cleaned.csv")

print(df.shape) 
display(df.head() ) 
display(df.info()) 


(2520751, 53)


,Destination Port,Flow Duration,Total Fwd Packets,Total Length of Fwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,...,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Max,Active Min,Idle Mean,Idle Max,Idle Min,Attack Type
0,22,1266342,41,2664,456,0,64.975610,109.864573,976,0,...,243,24,32,0.0,0,0,0.0,0,0,Normal Traffic
1,22,1319353,41,2664,456,0,64.975610,109.864573,976,0,...,243,24,32,0.0,0,0,0.0,0,0,Normal Traffic
2,22,160,1,0,0,0,0.000000,0.000000,0,0,...,243,0,32,0.0,0,0,0.0,0,0,Normal Traffic
3,22,1303488,41,2728,456,0,66.536585,110.129945,976,0,...,243,24,32,0.0,0,0,0.0,0,0,Normal Traffic
4,35396,77,1,0,0,0,0.000000,0.000000,0,0,...,290,0,32,0.0,0,0,0.0,0,0,Normal Traffic


<class 'pandas.DataFrame'>
RangeIndex: 2520751 entries, 0 to 2520750
Data columns (total 53 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   Destination Port             int64  
 1   Flow Duration                int64  
 2   Total Fwd Packets            int64  
 3   Total Length of Fwd Packets  int64  
 4   Fwd Packet Length Max        int64  
 5   Fwd Packet Length Min        int64  
 6   Fwd Packet Length Mean       float64
 7   Fwd Packet Length Std        float64
 8   Bwd Packet Length Max        int64  
 9   Bwd Packet Length Min        int64  
 10  Bwd Packet Length Mean       float64
 11  Bwd Packet Length Std        float64
 12  Flow Bytes/s                 float64
 13  Flow Packets/s               float64
 14  Flow IAT Mean                float64
 15  Flow IAT Std                 float64
 16  Flow IAT Max                 int64  
 17  Flow IAT Min                 int64  
 18  Fwd IAT Total                int64  
 19  Fwd IAT Mea

None

In [6]:
# kiem tra xem data đã clean chưa 
import numpy as np 

# kiem tra gia tri thieu 
missing = df.isna().sum()
print("Missing values:")
display(missing[missing > 0].sort_values(ascending=False))


Missing values:


Series([], dtype: int64)

# kiem tra gias tri isNan hay IsInf 

In [8]:
# kiem tra gia tri vo han 

numeric_cols = df.select_dtypes(include="number").columns 
inf_count = np.isinf(df[numeric_cols]).sum() 

print("Infinite values:")
display(inf_count[inf_count > 0].sort_values(ascending=False))

Infinite values:


Series([], dtype: int64)

In [13]:
# kiem tra nhan 
print("Attack classes:")
display(df["Attack Type"].value_counts())

Attack classes:


Attack Type
Normal Traffic    2095057
DoS                193745
DDoS               128014
Port Scanning       90694
Brute Force          9150
Web Attacks          2143
Bots                 1948
Name: count, dtype: int64

In [10]:
# kiem tra nhan co 1 du lieu 
constant_cols = df.nunique(dropna=False)
print("Constant columns:")
display(constant_cols[constant_cols <= 1])

Constant columns:


Series([], dtype: int64)

In [14]:
duplicate_count = df.duplicated().sum()
print("Duplicate rows:", duplicate_count)

Duplicate rows: 161


In [15]:
duplicate_mask = df.duplicated(keep=False)

print("Số dòng trùng:", duplicate_mask.sum())

display(
    df.loc[duplicate_mask, "Attack Type"]
      .value_counts(dropna=False)
)

Số dòng trùng: 322


Attack Type
Normal Traffic    322
Name: count, dtype: int64

In [16]:
numeric_cols = df.select_dtypes(include="number").columns

print("NaN:", df.isna().sum().sum())
print("Inf:", np.isinf(df[numeric_cols]).sum().sum())
print("Duplicate:", df.duplicated().sum())
print("Classes:")
display(df["Attack Type"].value_counts())

NaN: 0
Inf: 0
Duplicate: 161
Classes:


Attack Type
Normal Traffic    2095057
DoS                193745
DDoS               128014
Port Scanning       90694
Brute Force          9150
Web Attacks          2143
Bots                 1948
Name: count, dtype: int64

In [17]:
# kiem tra duplicate co khac label khong 
feature_cols = [
    col for col in df.columns
    if col != "Attack Type"
]

duplicate_features = df[
    df.duplicated(feature_cols, keep=False)
]

conflicts = (
    duplicate_features
    .groupby(feature_cols, dropna=False)["Attack Type"]
    .nunique()
)

print("Duplicate feature groups:", len(conflicts))
print("Conflicting-label groups:", (conflicts > 1).sum())

Duplicate feature groups: 858
Conflicting-label groups: 697


# neu Conflicting-label groups = 0 => moi co the xoa duplicate

In [19]:
n = len(df)

train = df.iloc[:int(n * 0.70)]
val = df.iloc[int(n * 0.70):int(n * 0.85)]
test = df.iloc[int(n * 0.85):]

for name, part in [
    ("TRAIN", train),
    ("VALIDATION", val),
    ("TEST", test)
]:
    print(f"\n{name}")
    print(part["Attack Type"].value_counts())
    print("\nPercentage:")
    print(part["Attack Type"].value_counts(normalize=True).mul(100).round(2))


TRAIN
Attack Type
Normal Traffic    1532576
DDoS               128014
Port Scanning       90694
Brute Force          9150
Web Attacks          2143
Bots                 1948
Name: count, dtype: int64

Percentage:
Attack Type
Normal Traffic    86.85
DDoS               7.25
Port Scanning      5.14
Brute Force        0.52
Web Attacks        0.12
Bots               0.11
Name: proportion, dtype: float64

VALIDATION
Attack Type
Normal Traffic    235241
DoS               142872
Name: count, dtype: int64

Percentage:
Attack Type
Normal Traffic    62.21
DoS               37.79
Name: proportion, dtype: float64

TEST
Attack Type
Normal Traffic    327240
DoS                50873
Name: count, dtype: int64

Percentage:
Attack Type
Normal Traffic    86.55
DoS               13.45
Name: proportion, dtype: float64


# chia deu du lieu cho can bang cai naof cung phai co du thu 

In [20]:
# chuan hoa nhan label 
df["Attack Type"] = (
    df["Attack Type"]
    .astype("string")
    .str.strip()
)

print(df["Attack Type"].value_counts())

Attack Type
Normal Traffic    2095057
DoS                193745
DDoS               128014
Port Scanning       90694
Brute Force          9150
Web Attacks          2143
Bots                 1948
Name: count, dtype: int64[pyarrow]


In [21]:
print(df["Attack Type"].unique())
print("Missing labels:", df["Attack Type"].isna().sum())

<ArrowStringArray>
['Normal Traffic',  'Port Scanning',    'Web Attacks',    'Brute Force',
           'DDoS',           'Bots',            'DoS']
Length: 7, dtype: string
Missing labels: 0


# kiem tra du lieu co duplicate thi loai bo 

In [22]:
before = len(df)

df = df.drop_duplicates().reset_index(drop=True)

print("Removed:", before - len(df))
print("Remaining:", len(df))

Removed: 161
Remaining: 2520590


In [23]:
constant_cols = [
    col for col in df.columns
    if df[col].nunique(dropna=False) <= 1
]

print("Constant columns:", constant_cols)

Constant columns: []


# du lieu sang buoc chia train/vali/test 

In [24]:
from sklearn.model_selection import train_test_split

target = "Attack Type"

# 70% train, 30% temporary
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df[target],
    random_state=42
)

# Chia 30% còn lại thành 15% validation và 15% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df[target],
    random_state=42
)

# Reset index
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

In [25]:
for name, part in [
    ("TRAIN", train_df),
    ("VALIDATION", val_df),
    ("TEST", test_df)
]:
    print(f"\n{name}")
    print("Shape:", part.shape)
    print(
        part[target]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )


TRAIN
Shape: (1764413, 53)
Attack Type
Normal Traffic    83.11
DoS                7.69
DDoS               5.08
Port Scanning       3.6
Brute Force        0.36
Web Attacks        0.09
Bots               0.08
Name: proportion, dtype: double[pyarrow]

VALIDATION
Shape: (378088, 53)
Attack Type
Normal Traffic    83.11
DoS                7.69
DDoS               5.08
Port Scanning       3.6
Brute Force        0.36
Web Attacks        0.09
Bots               0.08
Name: proportion, dtype: double[pyarrow]

TEST
Shape: (378089, 53)
Attack Type
Normal Traffic    83.11
DoS                7.69
DDoS               5.08
Port Scanning       3.6
Brute Force        0.36
Web Attacks        0.08
Bots               0.08
Name: proportion, dtype: double[pyarrow]


# tach feature va label ra => kt feature la so ? + 3 tap train/test/val co cung cau truc  

In [26]:
target = "Attack Type"

feature_cols = [
    col for col in train_df.columns
    if col != target
]

X_train = train_df[feature_cols]
y_train = train_df[target]

X_val = val_df[feature_cols]
y_val = val_df[target]

X_test = test_df[feature_cols]
y_test = test_df[target]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

X_train: (1764413, 52)
y_train: (1764413,)


In [27]:
non_numeric_cols = X_train.select_dtypes(
    exclude="number"
).columns.tolist()

print("Non-numeric columns:", non_numeric_cols)

Non-numeric columns: []


In [28]:
assert list(X_train.columns) == list(X_val.columns)
assert list(X_train.columns) == list(X_test.columns)

print("Feature structure: OK")

Feature structure: OK


In [29]:
from pathlib import Path

output_path = Path(
    r"D:\Python project\PROJECT\data\processed\cicids2017_processed.csv"
)

df.to_csv(output_path, index=False)

print("Saved to:", output_path)
print("Shape:", df.shape)

Saved to: D:\Python project\PROJECT\data\processed\cicids2017_processed.csv
Shape: (2520590, 53)


In [32]:
import pandas as pd

processed_path = r"D:\Python project\PROJECT\data\processed\cicids2017_processed.csv"

df_check = pd.read_csv(processed_path)

print(df_check.shape)
print(df_check.isna().sum().sum())
print(df_check["Attack Type"].value_counts())
print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(2520590, 53)
0
Attack Type
Normal Traffic    2094896
DoS                193745
DDoS               128014
Port Scanning       90694
Brute Force          9150
Web Attacks          2143
Bots                 1948
Name: count, dtype: int64
(1764413, 53)
(378088, 53)
(378089, 53)


# chia tap du lieu sau khi xu ly thanh 3 tap du lieu khac 

In [33]:
train_df.to_csv(r"D:\Python project\PROJECT\data\splits\train.csv", index=False)
val_df.to_csv(r"D:\Python project\PROJECT\data\splits\validation.csv", index=False)
test_df.to_csv(r"D:\Python project\PROJECT\data\splits\test.csv", index=False)

In [34]:
# xac nhan kich thuoc file 
from pathlib import Path

for filename in ["train.csv", "validation.csv", "test.csv"]:
    path = Path(r"D:\Python project\PROJECT") / filename
    print(filename, round(path.stat().st_size / (1024 ** 2), 2), "MB")

train.csv 480.51 MB
validation.csv 102.97 MB
test.csv 102.92 MB
